In [2]:
%load_ext autoreload
%autoreload 2

import torch

from utils import create_edge_loaders, create_hetero_graph
from gnn import model_factory
from train import train, evaluate, gnn_evaluate

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /nfs/roberts/project/pi_cmu2/jen55/ycrc_conda/envs/cpsc483/lib/python3.10/site-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /nfs/roberts/project/pi_cmu2/jen55/ycrc_conda/envs/cpsc483/lib/python3.10/site-packages/torch_cluster/_version_cuda.so
  import torch_geometric.typing
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /nfs/roberts/project/pi_cmu2/jen55/ycrc_conda

In [3]:
# Follow and activity data paths
FOLLOW_EDGELIST = 'data/higgs-social_network.edgelist'
ACTIVITY_EDGELIST = 'data/higgs-activity_time.txt'
target_interaction = "retweet"

torch.manual_seed(42)

In [3]:
model_path = train(
    model_name="sage", 
    target_interaction=target_interaction, 
    activity_data_path=ACTIVITY_EDGELIST, 
    follow_data_path=FOLLOW_EDGELIST,
    max_epochs=10,
    checkpoint_dir=f'checkpoints_{target_interaction}'
)

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jen55/.conda/envs/cpsc483/lib/python3.10/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud 

Starting training...
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 9: 100%|██████████| 1942/1942 [00:32<00:00, 59.88it/s, v_num=88, train_loss_step=0.666, train_acc_step=0.667, val_loss=0.835, val_auroc=0.608, val_acc=0.570, val_ap=0.452, val_recall=0.598, train_loss_epoch=0.621, train_acc_epoch=0.704]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 1942/1942 [00:32<00:00, 59.87it/s, v_num=88, train_loss_step=0.666, train_acc_step=0.667, val_loss=0.835, val_auroc=0.608, val_acc=0.570, val_ap=0.452, val_recall=0.598, train_loss_epoch=0.621, train_acc_epoch=0.704]
Training complete. Best model saved at: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_retweet/best-gnn-sage-retweetepoch=02-val_ap=0.5318.ckpt


In [4]:
# Evaluation
best_model_path = model_path
# best_model_path = "/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints/best-gnn-sage-followepoch=01-val_ap=0.8651.ckpt"
print(f"Evaluating best model from: {best_model_path}")

evaluate(
    checkpoint_path=best_model_path,
    model_name="sage",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    visualize_limit=0
)

Evaluating best model from: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_retweet/best-gnn-sage-retweetepoch=02-val_ap=0.5318.ckpt
Using device: cuda
Loading data...
Configuring CAPTUM Integrated Gradients...


Evaluating & Explaining (Captum):   0%|          | 0/416 [00:00<?, ?it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum): 100%|██████████| 416/416 [02:16<00:00,  3.05it/s]



--- Standard Metrics ---
Loss:   0.8144
AUROC:  0.7426
AP:     0.5402
Recall: 0.8165

--- Explanation Metrics (over 416 batches) ---
Saved visualization files to current directory.
Average Fidelity+: 0.7644 (Higher is better)
Average Fidelity-: 0.4351 (Lower is better)


In [5]:
# Using GNNExplainer
# best_model_path = "/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints/best-gnn-sage-followepoch=01-val_ap=0.8651.ckpt"
gnn_evaluate(
    checkpoint_path=best_model_path,
    model_name="sage",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    explain_limit=5
)

Using device: cuda
Loading data...

--- Evaluating Test Set ---


Evaluating: 100%|██████████| 416/416 [00:03<00:00, 114.30it/s]



--- Standard Metrics ---
Loss:  0.8119
AUROC: 0.7429
AP:    0.5423

--- Setting up Explainer (Option B) ---


Explaining:   0%|          | 0/5 [00:00<?, ?it/s]


--- Top Explanations for Target Edge 0 ---
  Top 5 important 'FL' edges:
    User 245 -> User 23 (Weight: 0.8177)
    User 19 -> User 23 (Weight: 0.8116)
    User 1 -> User 23 (Weight: 0.7898)
    User 220 -> User 23 (Weight: 0.7738)
    User 18 -> User 1 (Weight: 0.7616)
  Top 5 important 'RT' edges:
    User 288 -> User 23 (Weight: 0.8044)
    User 259 -> User 2 (Weight: 0.0831)
    User 280 -> User 5 (Weight: 0.0830)
    User 249 -> User 2 (Weight: 0.0830)
    User 274 -> User 5 (Weight: 0.0830)
  Top 5 important 'MT' edges:
    User 23 -> User 1 (Weight: 0.7225)
    User 316 -> User 23 (Weight: 0.1129)
    User 1 -> User 23 (Weight: 0.1051)
    User 317 -> User 23 (Weight: 0.1010)
    User 315 -> User 23 (Weight: 0.0879)
  Top 5 important 'RE' edges:
    User 320 -> User 2 (Weight: 0.9331)
    User 325 -> User 2 (Weight: 0.9331)
    User 267 -> User 2 (Weight: 0.9330)
    User 310 -> User 5 (Weight: 0.9317)
    User 23 -> User 1 (Weight: 0.8136)


...


--- Top Explanations for Target Edge 2 ---
  Top 5 important 'FL' edges:
    User 541 -> User 63 (Weight: 0.2864)
    User 901 -> User 63 (Weight: 0.2689)
    User 40 -> User 1 (Weight: 0.1887)
    User 51 -> User 1 (Weight: 0.1876)
    User 49 -> User 1 (Weight: 0.1831)
  Top 5 important 'RT' edges:
    User 43 -> User 1 (Weight: 0.3234)
    User 914 -> User 31 (Weight: 0.1055)
    User 668 -> User 46 (Weight: 0.1053)
    User 1013 -> User 52 (Weight: 0.1049)
    User 944 -> User 40 (Weight: 0.1048)
  Top 5 important 'MT' edges:
    User 62 -> User 0 (Weight: 0.2477)
    User 63 -> User 0 (Weight: 0.1954)
    User 1045 -> User 10 (Weight: 0.0982)
    User 1092 -> User 51 (Weight: 0.0976)
    User 1101 -> User 51 (Weight: 0.0975)
  Top 5 important 'RE' edges:
    User 63 -> User 0 (Weight: 0.2162)
    User 1051 -> User 25 (Weight: 0.0827)
    User 1047 -> User 24 (Weight: 0.0817)
    User 1047 -> User 24 (Weight: 0.0817)
    User 1056 -> User 33 (Weight: 0.0817)


Explaining:  80%|████████  | 4/5 [00:07<00:01,  1.82s/it]


--- Top Explanations for Target Edge 3 ---
  Top 5 important 'FL' edges:
    User 238 -> User 8 (Weight: 0.1500)
    User 551 -> User 30 (Weight: 0.1468)
    User 689 -> User 123 (Weight: 0.1463)
    User 654 -> User 38 (Weight: 0.1462)
    User 775 -> User 116 (Weight: 0.1462)
  Top 5 important 'RT' edges:
    User 71 -> User 1 (Weight: 0.0923)
    User 90 -> User 1 (Weight: 0.0922)
    User 82 -> User 1 (Weight: 0.0921)
    User 88 -> User 1 (Weight: 0.0921)
    User 1076 -> User 59 (Weight: 0.0920)
  Top 5 important 'MT' edges:
    User 97 -> User 1 (Weight: 0.0878)
    User 100 -> User 1 (Weight: 0.0878)
    User 119 -> User 1 (Weight: 0.0878)
    User 1100 -> User 86 (Weight: 0.0874)
    User 1104 -> User 102 (Weight: 0.0874)
  Top 5 important 'RE' edges:
    User 122 -> User 1 (Weight: 0.0788)
    User 97 -> User 1 (Weight: 0.0788)
    User 119 -> User 1 (Weight: 0.0788)
    User 1100 -> User 86 (Weight: 0.0788)
    User 108 -> User 1 (Weight: 0.0787)


Explaining: 100%|██████████| 5/5 [00:09<00:00,  1.87s/it]


--- Top Explanations for Target Edge 4 ---
  Top 5 important 'FL' edges:
    User 756 -> User 48 (Weight: 0.1511)
    User 39 -> User 1 (Weight: 0.1497)
    User 458 -> User 44 (Weight: 0.1484)
    User 589 -> User 34 (Weight: 0.1483)
    User 940 -> User 76 (Weight: 0.1475)
  Top 5 important 'RT' edges:
    User 1137 -> User 21 (Weight: 0.1081)
    User 1144 -> User 31 (Weight: 0.1076)
    User 79 -> User 0 (Weight: 0.1069)
    User 1123 -> User 21 (Weight: 0.1066)
    User 1110 -> User 11 (Weight: 0.1066)
  Top 5 important 'MT' edges:
    User 1214 -> User 11 (Weight: 0.1026)
    User 1228 -> User 11 (Weight: 0.1022)
    User 1238 -> User 21 (Weight: 0.1015)
    User 93 -> User 0 (Weight: 0.1014)
    User 1211 -> User 9 (Weight: 0.1012)
  Top 5 important 'RE' edges:
    User 115 -> User 0 (Weight: 0.0868)
    User 107 -> User 0 (Weight: 0.0857)
    User 110 -> User 0 (Weight: 0.0851)
    User 1291 -> User 32 (Weight: 0.0839)
    User 1276 -> User 76 (Weight: 0.0839)

--- Explanation

In [6]:
# Comparison with vanilla GNN
vanilla_model_path = train(
    model_name="simple",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    max_epochs=10,
    checkpoint_dir=f'checkpoints_{target_interaction}'
)

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jen55/.conda/envs/cpsc483/lib/python3.10/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_retweet exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type                   | Params | Mode  | FLOPs


Starting training...
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 9: 100%|██████████| 1942/1942 [00:37<00:00, 51.96it/s, v_num=89, train_loss_step=0.693, train_acc_step=0.667, val_loss=0.713, val_auroc=0.556, val_acc=0.697, val_ap=0.391, val_recall=0.132, train_loss_epoch=0.639, train_acc_epoch=0.783]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 1942/1942 [00:37<00:00, 51.96it/s, v_num=89, train_loss_step=0.693, train_acc_step=0.667, val_loss=0.713, val_auroc=0.556, val_acc=0.697, val_ap=0.391, val_recall=0.132, train_loss_epoch=0.639, train_acc_epoch=0.783]
Training complete. Best model saved at: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_retweet/best-gnn-simple-retweetepoch=01-val_ap=0.4004.ckpt


In [7]:
print(f"Evaluating vanilla GNN model from: {vanilla_model_path}")
evaluate(
    checkpoint_path=vanilla_model_path,
    model_name="simple",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    visualize_limit=5
)
gnn_evaluate(
    checkpoint_path=vanilla_model_path,
    model_name="simple",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    explain_limit=5
)

Evaluating vanilla GNN model from: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_retweet/best-gnn-simple-retweetepoch=01-val_ap=0.4004.ckpt
Using device: cuda
Loading data...
Configuring CAPTUM Integrated Gradients...


Evaluating & Explaining (Captum):   0%|          | 0/416 [00:00<?, ?it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum): 100%|██████████| 416/416 [05:49<00:00,  1.19it/s]



--- Standard Metrics ---
Loss:   0.7223
AUROC:  0.5737
AP:     0.4092
Recall: 0.1763

--- Explanation Metrics (over 416 batches) ---
Saved visualization files to current directory.
Average Fidelity+: 0.1875 (Higher is better)
Average Fidelity-: 0.1346 (Lower is better)
Using device: cuda
Loading data...

--- Evaluating Test Set ---


Evaluating: 100%|██████████| 416/416 [00:04<00:00, 83.43it/s]



--- Standard Metrics ---
Loss:  0.7214
AUROC: 0.5735
AP:    0.4101

--- Setting up Explainer (Option B) ---


...


--- Top Explanations for Target Edge 3 ---
  Top 5 important 'FL' edges:
    User 59 -> User 1 (Weight: 0.5921)
    User 42 -> User 1 (Weight: 0.5828)
    User 41 -> User 1 (Weight: 0.5786)
    User 50 -> User 1 (Weight: 0.5719)
    User 53 -> User 1 (Weight: 0.5712)
  Top 5 important 'RT' edges:
    User 87 -> User 1 (Weight: 0.4423)
    User 79 -> User 1 (Weight: 0.4325)
    User 89 -> User 1 (Weight: 0.4305)
    User 68 -> User 1 (Weight: 0.4291)
    User 72 -> User 1 (Weight: 0.4210)
  Top 5 important 'MT' edges:
    User 1078 -> User 30 (Weight: 0.6601)
    User 1146 -> User 104 (Weight: 0.6572)
    User 1147 -> User 104 (Weight: 0.6402)
    User 1146 -> User 104 (Weight: 0.5940)
    User 108 -> User 1 (Weight: 0.5928)
  Top 5 important 'RE' edges:
    User 1151 -> User 104 (Weight: 0.9310)
    User 1145 -> User 104 (Weight: 0.9296)
    User 1150 -> User 104 (Weight: 0.9284)
    User 1153 -> User 104 (Weight: 0.9239)
    User 123 -> User 1 (Weight: 0.4213)


Explaining:  80%|████████  | 4/5 [01:32<00:26, 26.13s/it]


--- Top Explanations for Target Edge 4 ---
  Top 5 important 'FL' edges:
    User 46 -> User 1 (Weight: 0.6140)
    User 38 -> User 1 (Weight: 0.6139)
    User 33 -> User 1 (Weight: 0.6059)
    User 48 -> User 1 (Weight: 0.6037)
    User 56 -> User 1 (Weight: 0.6014)
  Top 5 important 'RT' edges:
    User 1162 -> User 47 (Weight: 0.6045)
    User 1162 -> User 47 (Weight: 0.5839)
    User 1162 -> User 47 (Weight: 0.5797)
    User 569 -> User 50 (Weight: 0.5740)
    User 734 -> User 50 (Weight: 0.5708)
  Top 5 important 'MT' edges:
    User 103 -> User 0 (Weight: 0.5407)
    User 112 -> User 0 (Weight: 0.5402)
    User 1254 -> User 32 (Weight: 0.5338)
    User 97 -> User 0 (Weight: 0.5280)
    User 91 -> User 0 (Weight: 0.5254)
  Top 5 important 'RE' edges:
    User 1281 -> User 32 (Weight: 0.8927)
    User 1284 -> User 32 (Weight: 0.8322)
    User 1263 -> User 53 (Weight: 0.5440)
    User 560 -> User 44 (Weight: 0.5141)
    User 719 -> User 32 (Weight: 0.4407)


Explaining: 100%|██████████| 5/5 [01:47<00:00, 21.42s/it]


--- Explanation Metrics (over 5 edges) ---
Average Fidelity+: 0.0000 (Higher is better)
Average Fidelity-: 0.0000 (Lower is better)


In [4]:
# Ablation study: stripped GNN
stripped_model_path = train(
    model_name="stripped",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    max_epochs=10,
    checkpoint_dir=f'checkpoints_{target_interaction}'
)

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/jen55/.conda/envs/cpsc483/lib/python3.10/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud 

Starting training...
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jen55/.conda/envs/cpsc483/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 9: 100%|██████████| 1942/1942 [00:25<00:00, 75.85it/s, v_num=93, train_loss_step=0.578, train_acc_step=0.833, val_loss=0.687, val_auroc=0.526, val_acc=0.683, val_ap=0.366, val_recall=0.0554, train_loss_epoch=0.653, train_acc_epoch=0.731]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 1942/1942 [00:25<00:00, 75.84it/s, v_num=93, train_loss_step=0.578, train_acc_step=0.833, val_loss=0.687, val_auroc=0.526, val_acc=0.683, val_ap=0.366, val_recall=0.0554, train_loss_epoch=0.653, train_acc_epoch=0.731]
Training complete. Best model saved at: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_retweet/best-gnn-stripped-retweetepoch=01-val_ap=0.3921.ckpt


In [6]:
print(f"Evaluating stripped GNN model from: {stripped_model_path}")
evaluate(
    checkpoint_path=stripped_model_path,
    model_name="stripped",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    visualize_limit=5
)
gnn_evaluate(
    checkpoint_path=stripped_model_path,
    model_name="stripped",
    target_interaction=target_interaction,
    activity_data_path=ACTIVITY_EDGELIST,
    follow_data_path=FOLLOW_EDGELIST,
    explain_limit=5
)

Evaluating stripped GNN model from: /nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/checkpoints_retweet/best-gnn-stripped-retweetepoch=01-val_ap=0.3921.ckpt
Using device: cuda
Loading data...
Configuring CAPTUM Integrated Gradients...


Evaluating & Explaining (Captum):   0%|          | 0/416 [00:00<?, ?it/s]/nfs/roberts/scratch/pi_cmu2/jen55/GraphFinalProject/train/evaluate.py:164: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
Evaluating & Explaining (Captum): 100%|██████████| 416/416 [01:09<00:00,  5.97it/s]



--- Standard Metrics ---
Loss:   0.7023
AUROC:  0.5581
AP:     0.3994
Recall: 0.1346

--- Explanation Metrics (over 416 batches) ---
Saved visualization files to current directory.
Average Fidelity+: 0.0793 (Higher is better)
Average Fidelity-: 0.0649 (Lower is better)
Using device: cuda
Loading data...

--- Evaluating Test Set ---


Evaluating: 100%|██████████| 416/416 [00:01<00:00, 209.15it/s]



--- Standard Metrics ---
Loss:  0.7042
AUROC: 0.5579
AP:    0.3986

--- Setting up Explainer (Option B) ---


...


--- Top Explanations for Target Edge 2 ---
  Top 5 important 'FL' edges:
    User 46 -> User 1 (Weight: 0.0905)
    User 45 -> User 1 (Weight: 0.0898)
    User 55 -> User 1 (Weight: 0.0898)
    User 49 -> User 1 (Weight: 0.0896)
    User 43 -> User 1 (Weight: 0.0894)
  Top 5 important 'RT' edges:
    User 41 -> User 1 (Weight: 0.0724)
    User 980 -> User 8 (Weight: 0.0000)
    User 978 -> User 8 (Weight: 0.0000)
    User 981 -> User 8 (Weight: 0.0000)
    User 979 -> User 8 (Weight: 0.0000)
  Top 5 important 'MT' edges:
    User 63 -> User 0 (Weight: 0.9285)
    User 62 -> User 0 (Weight: 0.0724)
    User 1100 -> User 8 (Weight: 0.0000)
    User 133 -> User 7 (Weight: 0.0000)
    User 1099 -> User 7 (Weight: 0.0000)
  Top 5 important 'RE' edges:
    User 63 -> User 0 (Weight: 0.0723)
    User 1101 -> User 8 (Weight: 0.0000)
    User 1105 -> User 8 (Weight: 0.0000)
    User 1167 -> User 8 (Weight: 0.0000)
    User 1107 -> User 8 (Weight: 0.0000)


Explaining:  80%|████████  | 4/5 [00:03<00:00,  1.27it/s]


--- Top Explanations for Target Edge 3 ---
  Top 5 important 'FL' edges:
    User 3 -> User 0 (Weight: 0.0000)
    User 2 -> User 0 (Weight: 0.0000)
    User 4 -> User 0 (Weight: 0.0000)
    User 6 -> User 0 (Weight: 0.0000)
    User 5 -> User 0 (Weight: 0.0000)
  Top 5 important 'RT' edges:
    User 63 -> User 1 (Weight: 0.0000)
    User 62 -> User 1 (Weight: 0.0000)
    User 64 -> User 1 (Weight: 0.0000)
    User 66 -> User 1 (Weight: 0.0000)
    User 65 -> User 1 (Weight: 0.0000)
  Top 5 important 'MT' edges:
    User 93 -> User 1 (Weight: 0.0000)
    User 92 -> User 1 (Weight: 0.0000)
    User 94 -> User 1 (Weight: 0.0000)
    User 96 -> User 1 (Weight: 0.0000)
    User 95 -> User 1 (Weight: 0.0000)
  Top 5 important 'RE' edges:
    User 101 -> User 1 (Weight: 0.0000)
    User 113 -> User 1 (Weight: 0.0000)
    User 103 -> User 1 (Weight: 0.0000)
    User 105 -> User 1 (Weight: 0.0000)
    User 99 -> User 1 (Weight: 0.0000)


Explaining: 100%|██████████| 5/5 [00:03<00:00,  1.28it/s]


--- Top Explanations for Target Edge 4 ---
  Top 5 important 'FL' edges:
    User 3 -> User 0 (Weight: 0.0000)
    User 2 -> User 0 (Weight: 0.0000)
    User 4 -> User 0 (Weight: 0.0000)
    User 6 -> User 0 (Weight: 0.0000)
    User 5 -> User 0 (Weight: 0.0000)
  Top 5 important 'RT' edges:
    User 58 -> User 0 (Weight: 0.0000)
    User 57 -> User 0 (Weight: 0.0000)
    User 59 -> User 0 (Weight: 0.0000)
    User 61 -> User 0 (Weight: 0.0000)
    User 60 -> User 0 (Weight: 0.0000)
  Top 5 important 'MT' edges:
    User 83 -> User 0 (Weight: 0.0000)
    User 82 -> User 0 (Weight: 0.0000)
    User 84 -> User 0 (Weight: 0.0000)
    User 86 -> User 0 (Weight: 0.0000)
    User 85 -> User 0 (Weight: 0.0000)
  Top 5 important 'RE' edges:
    User 111 -> User 0 (Weight: 0.0000)
    User 110 -> User 0 (Weight: 0.0000)
    User 112 -> User 0 (Weight: 0.0000)
    User 1189 -> User 11 (Weight: 0.0000)
    User 1186 -> User 5 (Weight: 0.0000)

--- Explanation Metrics (over 5 edges) ---
Average F